# 📊 Bitcoin Price Prediction - Exploratory Data Analysis
> **Objective:** Perform comprehensive exploratory data analysis on Bitcoin historical price data to understand patterns, distributions, correlations, and time series characteristics.

This notebook covers:
- 📥 Data Loading & Cleaning
- 📈 Statistical Summary
- 🎨 Data Visualization (Price Trends, Volume, Correlations)
- 🔄 Time Series Decomposition
- 📉 Stationarity Analysis
- 📊 Rolling Statistics

## 1. Library Imports
Importing all necessary libraries for data manipulation, visualization, and statistical analysis.

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# EDA - Exploratory Data Analysis for Bitcoin Price Prediction
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plotting style for professional visuals
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

print("✓ Libraries imported successfully")

## 2. Data Loading
Loading the Bitcoin historical price dataset (2010-2024).

In [ ]:
# =============================================================================
# Load the dataset
# =============================================================================
df = pd.read_csv('../BTC-USD-Price-History-2010-2024.csv')

print(f"✓ Dataset loaded successfully")
print(f"  • Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"  • Columns: {list(df.columns)}")

## 3. Data Cleaning
The dataset contains price values with `$` and `,` characters. We need to clean these columns and convert them to numeric types for analysis.

In [ ]:
# =============================================================================
# Clean numeric columns - remove $ and , characters
# =============================================================================
cols_to_clean = ['Volume', 'Open', 'High', 'Low', 'Close']

for col in cols_to_clean:
    df[col] = (df[col]
               .astype(str)
               .str.replace('$', '', regex=False)
               .str.replace(',', '', regex=False)
               .str.replace(' ', '', regex=False)
               .astype(float))

# Convert Date column to datetime and set as index
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df = df.sort_values('Date')

print(f"✓ Data cleaning complete")
print(f"  • Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"  • Total records: {len(df)}")

## 4. Dataset Overview
A quick look at the data structure, data types, and missing values.

In [ ]:
# =============================================================================
# Dataset Information
# =============================================================================
print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)
print(f"\nShape: {df.shape}")

print("\n--- First 5 Records ---")
display(df.head())

print("\n--- Last 5 Records ---")
display(df.tail())

print("\n--- Data Types & Non-Null Counts ---")
print(df.info())

print("\n--- Missing Values ---")
print(df.isna().sum())

print("\n--- Missing Values % ---")
print((df.isna().sum() / len(df)) * 100)

## 5. Statistical Summary
Understanding the central tendency, dispersion, and distribution of our data. 

**Key Insights:**
- **High Price** ranges from $0.04 to $73,835 — showing massive growth
- **Volume** varies significantly, indicating changing market participation
- The large standard deviations indicate high volatility, characteristic of cryptocurrency markets

In [ ]:
# =============================================================================
# Descriptive Statistics
# =============================================================================
print("=" * 70)
print("DESCRIPTIVE STATISTICS")
print("=" * 70)

stats = df.describe()
display(stats)

# Additional statistics for more insights
print("\n--- Additional Statistics ---")
print(f"  • Price Range (High): ${stats.loc['min', 'High']:.2f} - ${stats.loc['max', 'High']:.2f}")
print(f"  • Mean Price (High): ${stats.loc['mean', 'High']:.2f}")
print(f"  • Median Price (High): ${stats.loc['50%', 'High']:.2f}")
print(f"  • Std Deviation (High): ${stats.loc['std', 'High']:.2f}")
print(f"  • CV (High) = {stats.loc['std', 'High'] / stats.loc['mean', 'High']:.2f} (Coefficient of Variation)")

## 6. Data Visualization

### 6.1 Price & Volume Trends Over Time

In [ ]:
# =============================================================================
# Price and Volume Trends
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot 1: High Price Over Time
axes[0].plot(df.index, df['High'], color='#f04747', linewidth=1.5, label='High Price')
axes[0].set_title('Bitcoin High Price Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Price (USD)', fontsize=12)
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].fill_between(df.index, df['High'], alpha=0.15, color='#f04747')

# Plot 2: Volume Over Time
axes[1].plot(df.index, df['Volume'], color='#4a90d9', linewidth=1.5, label='Volume')
axes[1].set_title('Bitcoin Trading Volume Over Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Volume', fontsize=12)
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)
axes[1].fill_between(df.index, df['Volume'], alpha=0.15, color='#4a90d9')

plt.tight_layout()
plt.show()
print("✓ Price and volume trend plots generated")

### 6.2 Correlation Analysis
A correlation heatmap helps us understand relationships between numerical features. 

**Key Observations:**
- High, Low, Open, Close prices are **perfectly correlated** (≈1.0) — expected since they move together
- Volume shows **moderate positive correlation** with prices — higher prices attract more trading activity
- This correlation structure validates using only 'High' price as our target variable

In [ ]:
# =============================================================================
# Correlation Heatmap
# =============================================================================
num_df = df.select_dtypes(include=['int64', 'float64'])
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f', 
            linewidths=0.5, square=True, ax=ax,
            annot_kws={'size': 11})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n--- Correlation with High Price ---")
print(corr['High'].sort_values(ascending=False).to_string())
print("\n✓ Correlation analysis complete")

### 6.3 Bitcoin Price Over Time (Detailed View)

In [ ]:
# =============================================================================
# Bitcoin Price Trend - Detailed View
# =============================================================================
fig, ax = plt.subplots(figsize=(18, 7))
ax.plot(df.index, df["High"], linestyle="-", linewidth=1.5, color='#f04747', label='Bitcoin High Price')
ax.fill_between(df.index, df["High"], alpha=0.2, color='#f04747')
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Price (USD)', fontsize=13)
ax.set_title('Bitcoin Price Over Time (2010-2024)', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.show()

print("✓ Detailed price trend plot generated")

### 6.4 Rolling Statistics
Rolling mean and standard deviation help identify trends and volatility patterns over time. The 12-period window smooths out short-term fluctuations.

In [ ]:
# =============================================================================
# Rolling Statistics
# =============================================================================
rolling_mean = df['High'].rolling(window=12).mean()
rolling_std = df['High'].rolling(window=12).std()

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(df['High'], color='cornflowerblue', linewidth=1.5, label='Original Price', alpha=0.7)
ax.plot(rolling_mean, color='firebrick', linewidth=2, label='Rolling Mean (12 periods)')
ax.plot(rolling_std, color='limegreen', linewidth=2, label='Rolling Std (12 periods)')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.set_title('Rolling Statistics - Bitcoin Price', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
plt.show()
print("✓ Rolling statistics plot generated")

## 7. Time Series Decomposition
Decomposing the time series into **Trend**, **Seasonal**, and **Residual** components. This helps us understand the underlying patterns in Bitcoin prices.

In [ ]:
# =============================================================================
# Time Series Decomposition
# =============================================================================
import statsmodels.api as sm

# Resample to daily frequency
ts = df['High'].resample('D').mean()
print(f"Time series shape after resampling: {len(ts)} daily observations")

# Perform additive decomposition
decomposition = sm.tsa.seasonal_decompose(ts.dropna(), model='additive')

fig, axes = plt.subplots(4, 1, figsize=(16, 12))

# Original
axes[0].plot(ts.index, ts.values, color='#2c3e50', linewidth=1)
axes[0].set_title('Original Bitcoin Price Series', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Trend
axes[1].plot(ts.index, decomposition.trend.values, color='#e74c3c', linewidth=1.5)
axes[1].set_title('Trend Component', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Seasonal
axes[2].plot(ts.index, decomposition.seasonal.values, color='#27ae60', linewidth=1)
axes[2].set_title('Seasonal Component', fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Residual
axes[3].plot(ts.index, decomposition.resid.values, color='#8e44ad', linewidth=1)
axes[3].set_title('Residual Component', fontweight='bold')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n--- Decomposition Insights ---")
print("  • Trend: Strong upward trend overall with some corrections")
print("  • Seasonality: Some cyclical patterns observed")
print("  • Residuals: Large spikes during volatile periods (crashes/rallies)")
print("✓ Time series decomposition complete")

## 8. Stationarity Analysis
Stationarity is a crucial concept in time series forecasting. A stationary series has constant mean, variance, and autocorrelation over time. We use the **Augmented Dickey-Fuller (ADF) test** to check stationarity.

**ADF Test:** 
- **Null Hypothesis (H₀):** Series has a unit root (non-stationary)
- **Alternate Hypothesis (H₁):** Series is stationary
- **p-value ≤ 0.05:** Reject H₀ → Series is stationary

In [ ]:
# =============================================================================
# Stationarity Test - Augmented Dickey-Fuller
# =============================================================================
from statsmodels.tsa.stattools import adfuller

# Test on original series
test_result = adfuller(ts.dropna())
print("=" * 70)
print("AUGMENTED DICKEY-FULLER TEST - Original Series")
print("=" * 70)
print(f"  • ADF Statistic: {test_result[0]:.6f}")
print(f"  • p-value: {test_result[1]:.6f}")
print(f"  • Critical Values:")
for key, value in test_result[4].items():
    print(f"      {key}: {value:.6f}")

if test_result[1] <= 0.05:
    print("\n  ✅ Conclusion: Reject H₀ → Series IS stationary (p-value ≤ 0.05)")
else:
    print("\n  ❌ Conclusion: Cannot reject H₀ → Series is NOT stationary (p-value > 0.05)")
    print("     We need to apply transformations to make it stationary.")

### 8.1 Making the Series Stationary
Since the original Bitcoin price series is non-stationary (p-value > 0.05), we apply:
1. **Log transformation** - to stabilize variance
2. **Differencing** - to remove trend and stabilize mean

In [ ]:
# =============================================================================
# Transform to Stationarity
# =============================================================================

# 1. Log transformation
ts_log = np.log(ts.dropna())
print("✓ Log transformation applied")

# 2. First-order differencing
ts_diff = ts_log.diff().dropna()
print("✓ First-order differencing applied")

# Test on transformed series
test_result_diff = adfuller(ts_diff)
print("\n" + "=" * 70)
print("AUGMENTED DICKEY-FULLER TEST - After Transformation")
print("=" * 70)
print(f"  • ADF Statistic: {test_result_diff[0]:.6f}")
print(f"  • p-value: {test_result_diff[1]:.6f}")

if test_result_diff[1] <= 0.05:
    print("  \n  ✅ Series IS stationary after log transformation + differencing!")
else:
    print("  \n  ⚠️  Series still not stationary - may need higher order differencing")

### 8.2 Visualizing the Transformed Series

In [ ]:
# =============================================================================
# Visualize Transformed Series with Rolling Statistics
# =============================================================================
rolling_mean_diff = ts_diff.rolling(12).mean()
rolling_std_diff = ts_diff.rolling(12).std()

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Original vs Log-transformed
axes[0].plot(ts.index, ts.values, color='blue', alpha=0.6, label='Original Price', linewidth=1)
axes[0].plot(ts_log.index, ts_log.values, color='orange', alpha=0.8, label='Log-transformed Price', linewidth=1)
axes[0].set_title('Original vs Log-transformed Bitcoin Price', fontweight='bold')
axes[0].set_ylabel('Price / Log(Price)', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Differenced series
axes[1].plot(ts_diff.index, ts_diff.values, color='red', linewidth=1, label='Differenced Log Price')
axes[1].plot(rolling_mean_diff.index, rolling_mean_diff.values, color='blue', linewidth=2, label='Rolling Mean (12)')
axes[1].plot(rolling_std_diff.index, rolling_std_diff.values, color='green', linewidth=2, label='Rolling Std (12)')
axes[1].set_title('Stationary Series - Differenced Log Price with Rolling Stats', fontweight='bold')
axes[1].set_ylabel('Differenced Log(Price)', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✓ Transformed series visualization complete")

## 9. EDA Summary & Key Insights

| Aspect | Finding |
|--------|---------|
| **Data Range** | 2010 to 2024 (14+ years of data) |
| **Trend** | Strong upward trend with significant volatility |
| **Correlation** | Price features are highly correlated; Volume moderately correlated with price |
| **Stationarity** | Original series is non-stationary; log+differencing achieves stationarity |
| **Volatility** | High volatility with periods of extreme price movements |
| **Seasonality** | Some cyclical patterns detected but not dominant |

> **Next Step:** Proceed to **Feature Engineering** notebook to create features for modeling.